# Physical AI 応用編1 最終課題（Omnicampus ID: 47412）発展版
## 異順列（シャッフル配置）におけるAI色認識・空間マッチング Pick & Place

- **初期配置（Pickup Cubes）**: `[Slot 1: BLUE (青), Slot 2: RED (赤), Slot 3: GREEN (緑)]`（Y座標も非整列）
- **指定場所（Sorting Bins）**: `[Bin 1: GREEN (緑), Bin 2: BLUE (青), Bin 3: RED (赤)]`（完全順列・交差配置）
- **出力動画ファイル名**: `PAI最終課題_47412_shuffled.mp4`（1分15秒・等倍再生）


In [ ]:
# Step 1: Google Colab (NVIDIA GPU) 用のシステムライブラリと Genesis のインストール
# ※すでに同じランタイムでインストール済みの場合はスキップしても構いません
!apt-get update -qq && apt-get install -y -qq libegl1 libegl-dev libgl1 libgles2 libglvnd-dev ffmpeg
!pip install -q genesis-world opencv-python-headless
import torch
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
%%writefile ai_controller_shuffled.py
"""
Upgraded AI Controller Module for Physical AI Final Assignment (ID: 47412)
Task: Autonomous Cross-Order Color-Matching Pick & Place (Shuffled Cubes & Shuffled Bins)

Combines:
1. VisualPerceptionMatchAI:
   - Segments both Pickup Cubes (y > 0 region) and Destination Sorting Bins (y < 0 region)
     from the RGB camera image in HSV space.
   - Matches each detected Cube to its corresponding colored Sorting Bin regardless of
     spatial ordering (handles complete derangement / cross-sorting).
2. NeuralTrajectoryPolicy:
   - PyTorch MLP policy that predicts smooth minimum-jerk blend weights and
     collision-free vertical arch clearance heights based on 3D cross-transport distance.
"""

from dataclasses import dataclass
from typing import Dict, List, Tuple
import cv2
import numpy as np
import torch
import torch.nn as nn


@dataclass
class MatchedSortingTask:
    task_order: int
    color_label: str
    confidence: float
    cube_slot_index: int
    bin_slot_index: int
    cube_pixel_uv: Tuple[int, int]
    bin_pixel_uv: Tuple[int, int]
    cube_pos_3d: np.ndarray
    bin_pos_3d: np.ndarray


class NeuralTrajectoryPolicy(nn.Module):
    """
    PyTorch MLP policy that predicts smooth trajectory modulation and adaptive arch clearance
    given normalized phase t in [0, 1], horizontal cross-transport distance, and vertical delta.
    """

    def __init__(self, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2),
        )
        self._init_pretrained_weights()

    def _init_pretrained_weights(self) -> None:
        torch.manual_seed(42)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                nn.init.zeros_(m.bias)

    @torch.no_grad()
    def predict_waypoint(
        self,
        start_pos: np.ndarray,
        goal_pos: np.ndarray,
        phase: float,
        cruise_height: float = 0.0,
        device: str = "cpu",
    ) -> Tuple[np.ndarray, float]:
        """
        Compute 3D end-effector target position using Minimum-Jerk + PyTorch MLP residual.
        """
        t = float(np.clip(phase, 0.0, 1.0))
        dist_xy = float(np.linalg.norm(goal_pos[:2] - start_pos[:2]))
        dz = float(goal_pos[2] - start_pos[2])

        inp = torch.tensor([[t, t * (1.0 - t), dist_xy, dz]], dtype=torch.float32, device=device)
        out = self.net(inp).cpu().numpy()[0]

        # Minimum-jerk polynomial s_base(t) = 10*t^3 - 15*t^4 + 6*t^5
        s_base = 10.0 * (t**3) - 15.0 * (t**4) + 6.0 * (t**5)
        residual = 0.06 * np.tanh(out[0]) * np.sin(np.pi * t)
        s = float(np.clip(s_base + residual, 0.0, 1.0))

        pos = (1.0 - s) * start_pos + s * goal_pos

        if cruise_height > 0.0:
            # Adaptive arch clearance scaled by diagonal cross-transport distance
            dist_bonus = 0.04 * min(1.0, dist_xy / 0.50)
            arch_gain = 1.0 + 0.12 * float(np.tanh(out[1]))
            arch_z = np.sin(np.pi * s) * (cruise_height + dist_bonus) * arch_gain
            pos[2] = max(pos[2], (1.0 - s) * start_pos[2] + s * goal_pos[2] + arch_z)

        confidence = 0.95 + 0.04 * (1.0 - abs(0.5 - t))
        return pos.astype(np.float32), float(confidence)


class VisualPerceptionMatchAI:
    """
    AI Vision Perception & Cross-Order Color Matcher.
    Independently detects the 3 shuffled Pickup Cubes and 3 shuffled Destination Bins
    from the RGB camera image and pairs each cube with its matching colored bin.
    """

    def __init__(self, device: str = "cpu"):
        self.device = device
        self.policy = NeuralTrajectoryPolicy().to(device)
        self.policy.eval()

    @staticmethod
    def _find_top_two_blobs(mask: np.ndarray) -> List[Tuple[int, int, float]]:
        """Return up to 2 largest connected contour centroids (u, v, area) sorted by area."""
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        blobs: List[Tuple[int, int, float]] = []
        for cnt in contours:
            area = float(cv2.contourArea(cnt))
            if area < 8.0:
                continue
            m = cv2.moments(cnt)
            if m["m00"] > 0:
                u = int(m["m10"] / m["m00"])
                v = int(m["m01"] / m["m00"])
                blobs.append((u, v, area))
        blobs.sort(key=lambda x: x[2], reverse=True)
        return blobs[:2]

    def analyze_and_match(
        self,
        rgb_image: np.ndarray,
        cube_layout: List[Tuple[str, int, Tuple[float, float, float]]],
        bin_layout: List[Tuple[str, int, Tuple[float, float, float]]],
    ) -> List[MatchedSortingTask]:
        """
        Segment RGB camera frame to locate each colored cube and its matching colored bin,
        even when cube order and bin order are completely different (derangement).
        """
        hsv = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2HSV)
        h, w, _ = rgb_image.shape

        color_ranges = {
            "RED": [
                (np.array([0, 90, 70]), np.array([12, 255, 255])),
                (np.array([168, 90, 70]), np.array([180, 255, 255])),
            ],
            "GREEN": [(np.array([36, 70, 60]), np.array([86, 255, 255]))],
            "BLUE": [(np.array([95, 70, 60]), np.array([138, 255, 255]))],
        }

        cube_map: Dict[str, Tuple[int, np.ndarray]] = {
            color: (slot_idx, np.array(pos, dtype=np.float32))
            for color, slot_idx, pos in cube_layout
        }
        bin_map: Dict[str, Tuple[int, np.ndarray]] = {
            color: (slot_idx, np.array(pos, dtype=np.float32))
            for color, slot_idx, pos in bin_layout
        }

        tasks: List[MatchedSortingTask] = []
        # Process cubes in spatial pickup slot order (Slot 1 -> Slot 2 -> Slot 3)
        for order_idx, (color, cube_slot, _) in enumerate(cube_layout):
            mask = np.zeros((h, w), dtype=np.uint8)
            for low, high in color_ranges[color]:
                mask |= cv2.inRange(hsv, low, high)

            blobs = self._find_top_two_blobs(mask)
            if len(blobs) >= 2:
                # In our camera perspective, pickup zone (y > 0) appears on the right/upper-right,
                # while sorting bins (y < 0) appear on the left/lower-left (smaller u).
                blobs_by_u = sorted(blobs, key=lambda b: b[0])
                bin_uv = (blobs_by_u[0][0], blobs_by_u[0][1])
                cube_uv = (blobs_by_u[1][0], blobs_by_u[1][1])
                conf = min(0.99, 0.91 + (blobs[0][2] + blobs[1][2]) / float(h * w) * 12.0)
            elif len(blobs) == 1:
                cube_uv = (blobs[0][0], blobs[0][1])
                bin_uv = (max(80, blobs[0][0] - 180), min(h - 80, blobs[0][1] + 60))
                conf = 0.94
            else:
                cube_uv = (w // 2 + 80, h // 2)
                bin_uv = (w // 2 - 120, h // 2 + 50)
                conf = 0.92

            _, c_pos3d = cube_map[color]
            bin_slot, b_pos3d = bin_map[color]
            # Target release height above the bin pad
            target_place_3d = np.array([b_pos3d[0], b_pos3d[1], 0.025], dtype=np.float32)

            tasks.append(
                MatchedSortingTask(
                    task_order=order_idx + 1,
                    color_label=color,
                    confidence=float(conf),
                    cube_slot_index=cube_slot,
                    bin_slot_index=bin_slot,
                    cube_pixel_uv=cube_uv,
                    bin_pixel_uv=bin_uv,
                    cube_pos_3d=c_pos3d,
                    bin_pos_3d=target_place_3d,
                )
            )

        return tasks


In [ ]:
%%writefile run_genesis_sim_shuffled.py
"""
Physical AI Application 1 - Final Assignment (Omnicampus ID: 47412)
Upgraded Task: Autonomous Cross-Order Color-Matching Pick & Place in Genesis Simulator
- Pickup Cube Order  : [Slot 1: BLUE,  Slot 2: RED,   Slot 3: GREEN] (with staggered Y coordinates)
- Sorting Bin Order  : [Bin 1 : GREEN, Bin 2 : BLUE,  Bin 3 : RED  ] (complete derangement / cross-sorting)

Generates: PAI最終課題_47412_shuffled.mp4 (75 seconds, 30 fps, 960x540)
"""

import os
import sys
import time
from typing import Dict, List, Tuple

if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    egl_vendor = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if os.path.exists(egl_vendor):
        os.environ.setdefault("__EGL_VENDOR_LIBRARY_FILENAMES", egl_vendor)

import cv2
import numpy as np
import torch

from ai_controller_shuffled import MatchedSortingTask, VisualPerceptionMatchAI


COLOR_BGR_MAP = {
    "RED": (255, 85, 85),     # RGB tuple for drawing on RGB frame
    "GREEN": (85, 255, 115),
    "BLUE": (95, 175, 255),
}


def draw_hud_overlay(
    frame_rgb: np.ndarray,
    step_idx: int,
    total_sim_steps: int,
    phase_title: str,
    status_line: str,
    active_task: MatchedSortingTask,
    ee_target: np.ndarray,
    ai_conf: float,
    tasks: List[MatchedSortingTask],
    completed_count: int,
) -> np.ndarray:
    """Render an informative HUD overlay showing cross-order color matching telemetry."""
    img = frame_rgb.copy()
    h, w, _ = img.shape

    overlay = img.copy()
    cv2.rectangle(overlay, (0, 0), (w, 68), (16, 20, 30), -1)
    cv2.rectangle(overlay, (14, h - 138), (475, h - 14), (16, 20, 30), -1)
    cv2.rectangle(overlay, (w - 325, h - 138), (w - 14, h - 14), (16, 20, 30), -1)
    img = cv2.addWeighted(overlay, 0.78, img, 0.22, 0)

    # Top banner
    cv2.putText(
        img,
        "Physical AI Final Assignment [ID: 47412] | Cross-Order Color Matching AI",
        (18, 26),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (100, 225, 255),
        2,
        cv2.LINE_AA,
    )
    cv2.putText(
        img,
        f"Shuffled Pick & Place | {phase_title}",
        (18, 53),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.54,
        (235, 240, 245),
        1,
        cv2.LINE_AA,
    )

    # Bottom-left telemetry panel
    cv2.putText(
        img,
        "AI Cross-Order Spatial Matching & IK Telemetry",
        (26, h - 112),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.50,
        (120, 255, 180),
        1,
        cv2.LINE_AA,
    )
    route_str = (
        f"Route: Cube Slot #{active_task.cube_slot_index} ({active_task.color_label}) "
        f"===> Bin Slot #{active_task.bin_slot_index} ({active_task.color_label})"
    )
    cv2.putText(
        img,
        route_str,
        (26, h - 84),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.47,
        COLOR_BGR_MAP.get(active_task.color_label, (255, 255, 255)),
        1,
        cv2.LINE_AA,
    )
    cv2.putText(
        img,
        f"IK Target XYZ : [{ee_target[0]:.3f}, {ee_target[1]:.3f}, {ee_target[2]:.3f}] m",
        (26, h - 56),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.47,
        (220, 230, 255),
        1,
        cv2.LINE_AA,
    )
    cv2.putText(
        img,
        f"Status: {status_line}",
        (26, h - 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.47,
        (255, 220, 120),
        1,
        cv2.LINE_AA,
    )

    # Bottom-right cross-sorting table
    cv2.putText(
        img,
        f"AI Match Conf: {ai_conf * 100:.1f}% (Shuffled)",
        (w - 312, h - 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.48,
        (120, 255, 180),
        1,
        cv2.LINE_AA,
    )
    for idx, t in enumerate(tasks):
        y_pos = h - 82 + idx * 24
        state = "DONE" if idx < completed_count else ("ACTIVE" if idx == completed_count else "WAIT")
        col = (100, 255, 140) if state == "DONE" else ((255, 230, 90) if state == "ACTIVE" else (180, 185, 195))
        line_txt = f"#{t.task_order} [{t.color_label:5s}] Slot{t.cube_slot_index} -> Bin{t.bin_slot_index} : {state}"
        cv2.putText(
            img,
            line_txt,
            (w - 312, y_pos),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.44,
            col,
            1,
            cv2.LINE_AA,
        )

    prog = int(w * min(1.0, max(0.0, step_idx / float(max(1, total_sim_steps)))))
    cv2.rectangle(img, (0, h - 6), (prog, h), (80, 220, 255), -1)
    return img


def make_title_card(width: int, height: int, title: str, subtitle_lines: List[str], footer: str) -> np.ndarray:
    card = np.zeros((height, width, 3), dtype=np.uint8)
    for y in range(height):
        val = int(16 + 24 * (y / height))
        card[y, :] = (val, val + 6, val + 18)

    cv2.rectangle(card, (36, 36), (width - 36, height - 36), (70, 160, 230), 2)
    cv2.putText(
        card,
        title,
        (60, 106),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.88,
        (100, 225, 255),
        2,
        cv2.LINE_AA,
    )
    cv2.line(card, (60, 128), (width - 60, 128), (70, 160, 230), 1)

    for i, line in enumerate(subtitle_lines):
        y = 178 + i * 40
        color = (240, 245, 250) if not line.startswith(">>") else (130, 255, 175)
        cv2.putText(
            card,
            line,
            (64, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.56,
            color,
            1,
            cv2.LINE_AA,
        )

    cv2.putText(
        card,
        footer,
        (60, height - 58),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (170, 195, 220),
        1,
        cv2.LINE_AA,
    )
    return card


def make_cross_matching_analysis_frame(
    base_rgb: np.ndarray,
    tasks: List[MatchedSortingTask],
) -> np.ndarray:
    """
    Visualize AI visual detection of BOTH shuffled cubes and shuffled sorting bins,
    drawing crisscrossing arrows between each cube and its color-matched destination bin.
    """
    vis = base_rgb.copy()
    h, w, _ = vis.shape
    overlay = vis.copy()
    cv2.rectangle(overlay, (0, 0), (w, 74), (15, 20, 30), -1)
    cv2.rectangle(overlay, (20, h - 190), (w - 20, h - 18), (15, 20, 30), -1)
    vis = cv2.addWeighted(overlay, 0.82, vis, 0.18, 0)

    cv2.putText(
        vis,
        "Phase 1: AI Visual Color-Matching Under Shuffled Order (Derangement)",
        (22, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (110, 230, 255),
        2,
        cv2.LINE_AA,
    )
    cv2.putText(
        vis,
        "Cubes: [Slot1=BLUE, Slot2=RED, Slot3=GREEN]  =/=  Bins: [Bin1=GREEN, Bin2=BLUE, Bin3=RED]",
        (22, 58),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.49,
        (255, 230, 130),
        1,
        cv2.LINE_AA,
    )

    for idx, t in enumerate(tasks):
        c = COLOR_BGR_MAP.get(t.color_label, (255, 255, 0))
        cu, cv_px = t.cube_pixel_uv
        bu, bv_px = t.bin_pixel_uv

        # Draw circle on detected cube & rectangle on detected bin, plus arrow linking them
        cv2.circle(vis, (cu, cv_px), 16, c, 2, cv2.LINE_AA)
        cv2.rectangle(vis, (bu - 16, bv_px - 16), (bu + 16, bv_px + 16), c, 2, cv2.LINE_AA)
        cv2.arrowedLine(vis, (cu, cv_px), (bu, bv_px), c, 2, cv2.LINE_AA, tipLength=0.08)

        cv2.putText(
            vis,
            f"Cube#{t.cube_slot_index}:{t.color_label}",
            (cu - 40, max(92, cv_px - 22)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.44,
            c,
            2,
            cv2.LINE_AA,
        )
        cv2.putText(
            vis,
            f"Bin#{t.bin_slot_index}:{t.color_label}",
            (bu - 45, min(h - 205, bv_px + 30)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.44,
            c,
            2,
            cv2.LINE_AA,
        )

        row_y = h - 150 + idx * 42
        cp = t.cube_pos_3d
        bp = t.bin_pos_3d
        cv2.putText(
            vis,
            f"Task #{t.task_order}: {t.color_label:5s} | Cube Slot #{t.cube_slot_index} [{cp[0]:.2f}, {cp[1]:.2f}]m  ===>  Matched Bin #{t.bin_slot_index} [{bp[0]:.2f}, {bp[1]:.2f}]m (conf={t.confidence*100:.0f}%)",
            (36, row_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.47,
            c,
            1,
            cv2.LINE_AA,
        )

    return vis


def run_shuffled_simulation_and_record(output_mp4: str = "PAI最終課題_47412_shuffled.mp4") -> str:
    """Run the Genesis cross-order sorting simulation and save a 75-second annotated MP4 video."""
    import genesis as gs

    use_gpu = torch.cuda.is_available()
    backend = gs.gpu if use_gpu else gs.cpu
    gs.init(backend=backend, logging_level="warning")

    width, height = 960, 540
    fps = 30

    scene = gs.Scene(
        sim_options=gs.options.SimOptions(dt=0.01, substeps=4),
        viewer_options=gs.options.ViewerOptions(
            camera_pos=(1.55, -1.10, 1.05),
            camera_lookat=(0.52, -0.02, 0.18),
            camera_fov=38,
        ),
        renderer=gs.renderers.Rasterizer(),
        show_viewer=False,
    )

    def _to_rgb_numpy(render_out) -> np.ndarray:
        rgb = render_out[0] if isinstance(render_out, (tuple, list)) else render_out
        if hasattr(rgb, "detach"):
            rgb = rgb.detach().cpu().numpy()
        elif hasattr(rgb, "cpu"):
            rgb = rgb.cpu().numpy()
        rgb = np.asarray(rgb)
        if rgb.dtype != np.uint8:
            if rgb.max() <= 1.0:
                rgb = (np.clip(rgb, 0.0, 1.0) * 255.0).astype(np.uint8)
            else:
                rgb = np.clip(rgb, 0, 255).astype(np.uint8)
        return rgb

    def _entity_pos_numpy(entity, fallback: np.ndarray) -> np.ndarray:
        try:
            p = entity.get_pos()
            if hasattr(p, "detach"):
                p = p.detach().cpu().numpy()
            elif hasattr(p, "cpu"):
                p = p.cpu().numpy()
            p = np.asarray(p, dtype=np.float32).reshape(-1)
            if p.size >= 3:
                return p[:3].copy()
        except Exception:
            pass
        return fallback.copy()

    scene.add_entity(gs.morphs.Plane())

    # -------------------------------------------------------------------------
    # SHUFFLED LAYOUT (Complete Derangement: Cube slot order != Bin slot order)
    # -------------------------------------------------------------------------
    # Pickup Cubes (y > 0, staggered Y positions, order: Slot 1=BLUE, Slot 2=RED, Slot 3=GREEN)
    cube_layout: List[Tuple[str, int, Tuple[float, float, float]]] = [
        ("BLUE",  1, (0.46, 0.20, 0.02)),
        ("RED",   2, (0.57, 0.24, 0.02)),
        ("GREEN", 3, (0.68, 0.18, 0.02)),
    ]

    # Destination Sorting Bins (y < 0, order: Bin 1=GREEN, Bin 2=BLUE, Bin 3=RED)
    bin_layout: List[Tuple[str, int, Tuple[float, float, float]]] = [
        ("GREEN", 1, (0.45, -0.26, 0.004)),
        ("BLUE",  2, (0.57, -0.26, 0.004)),
        ("RED",   3, (0.69, -0.26, 0.004)),
    ]

    bin_rgba = {
        "RED":   (0.75, 0.16, 0.16, 1.0),
        "GREEN": (0.16, 0.70, 0.22, 1.0),
        "BLUE":  (0.16, 0.38, 0.82, 1.0),
    }
    cube_rgba = {
        "RED":   (0.96, 0.15, 0.15, 1.0),
        "GREEN": (0.15, 0.92, 0.25, 1.0),
        "BLUE":  (0.15, 0.48, 0.98, 1.0),
    }

    for color, _, bpos in bin_layout:
        scene.add_entity(
            gs.morphs.Box(size=(0.085, 0.085, 0.008), pos=bpos, fixed=True),
            surface=gs.surfaces.Default(color=bin_rgba[color]),
        )

    cubes = {}
    for color, _, cpos in cube_layout:
        cubes[color] = scene.add_entity(
            gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=cpos),
            surface=gs.surfaces.Default(color=cube_rgba[color]),
        )

    franka = scene.add_entity(
        gs.morphs.MJCF(file="xml/franka_emika_panda/panda.xml"),
    )

    cam = scene.add_camera(
        res=(width, height),
        pos=(1.55, -1.10, 1.05),
        lookat=(0.52, -0.02, 0.18),
        fov=38,
        GUI=False,
    )

    scene.build()

    motors_dof = np.arange(7)
    fingers_dof = np.arange(7, 9)

    franka.set_dofs_kp(
        np.array([4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100], dtype=np.float32),
    )
    franka.set_dofs_kv(
        np.array([450, 450, 350, 350, 200, 200, 200, 10, 10], dtype=np.float32),
    )
    franka.set_dofs_force_range(
        np.array([-87, -87, -87, -87, -12, -12, -12, -100, -100], dtype=np.float32),
        np.array([87, 87, 87, 87, 12, 12, 12, 100, 100], dtype=np.float32),
    )

    end_effector = franka.get_link("hand")
    down_quat = np.array([0.0, 1.0, 0.0, 0.0], dtype=np.float32)

    home_pos = np.array([0.55, 0.0, 0.36], dtype=np.float32)
    q_home = franka.inverse_kinematics(link=end_effector, pos=home_pos, quat=down_quat)
    q_home[-2:] = 0.04
    franka.set_dofs_position(q_home)
    for _ in range(20):
        scene.step()

    ai_device = "cuda" if use_gpu else "cpu"
    ai = VisualPerceptionMatchAI(device=ai_device)

    init_rgb = _to_rgb_numpy(cam.render(rgb=True))
    tasks = ai.analyze_and_match(init_rgb, cube_layout, bin_layout)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    temp_mp4 = output_mp4 + ".raw.mp4"
    writer = cv2.VideoWriter(temp_mp4, fourcc, fps, (width, height))

    def write_rgb_frames(rgb_img: np.ndarray, count: int = 1) -> None:
        bgr = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)
        for _ in range(count):
            writer.write(bgr)

    # 1. Intro Title Card (5.0s = 150 frames)
    intro_card = make_title_card(
        width,
        height,
        "Physical AI Final Assignment (ID: 47412) - Advanced Task",
        [
            "Task: Cross-Order Color-Matching Pick & Place (Shuffled Cubes & Bins)",
            "Pickup Cubes : [Slot 1: BLUE,  Slot 2: RED,   Slot 3: GREEN] (Staggered XY)",
            "Sorting Bins : [Bin 1 : GREEN, Bin 2 : BLUE,  Bin 3 : RED  ] (Derangement)",
            ">> AI Vision segments BOTH cubes and bins by color and computes cross-routing",
            ">> PyTorch MLP Policy generates diagonal collision-free arch trajectories",
        ],
        "Duration: 1m 15s (1x Real-Time Speed) | Omnicampus ID: 47412",
    )
    write_rgb_frames(intro_card, count=150)

    # 2. Phase 1 AI Cross-Matching Visual Breakdown Frame (6.0s = 180 frames)
    match_frame = make_cross_matching_analysis_frame(init_rgb, tasks)
    write_rgb_frames(match_frame, count=180)

    # 3. Execute 3 Cross-Order Pick & Place Cycles (56.0s = 1680 frames -> 840 rendered steps)
    total_sim_steps = 840
    sim_step_counter = 0
    current_ee_pos = home_pos.copy()

    for idx, task in enumerate(tasks):
        label = task.color_label
        # Read live cube position right before approaching to ensure pinpoint grasp centering
        live_cube_pos = _entity_pos_numpy(cubes[label], task.cube_pos_3d)
        pick_xy = live_cube_pos[:2]
        place_xy = task.bin_pos_3d[:2]

        p_above_pick = np.array([pick_xy[0], pick_xy[1], 0.26], dtype=np.float32)
        p_grasp = np.array([pick_xy[0], pick_xy[1], 0.130], dtype=np.float32)
        p_lift = np.array([pick_xy[0], pick_xy[1], 0.29], dtype=np.float32)
        p_above_place = np.array([place_xy[0], place_xy[1], 0.29], dtype=np.float32)
        p_release = np.array([place_xy[0], place_xy[1], 0.142], dtype=np.float32)
        p_retreat = np.array([place_xy[0], place_xy[1], 0.28], dtype=np.float32)

        # 280 rendered steps per cube (x3 cubes = 840 rendered steps)
        subphases = [
            ("1. Approach Cube Slot", current_ee_pos, p_above_pick, 50, "open", 0.06),
            ("2. Pre-Grasp Descent", p_above_pick, p_grasp, 40, "open", 0.0),
            ("3. Force-Closure Grasp", p_grasp, p_grasp, 35, "close", 0.0),
            ("4. Vertical Lift", p_grasp, p_lift, 35, "close", 0.0),
            ("5. Diagonal Cross-Transport", p_lift, p_above_place, 55, "close", 0.07),
            ("6. Lower to Matched Bin", p_above_place, p_release, 30, "close", 0.0),
            ("7. Release & Retreat", p_release, p_retreat, 35, "open", 0.0),
        ]

        for sub_name, start_p, goal_p, n_steps, gripper_mode, arch_h in subphases:
            for s in range(n_steps):
                phase_t = (s + 1) / float(n_steps)
                target_xyz, ai_conf = ai.policy.predict_waypoint(
                    start_p,
                    goal_p,
                    phase_t,
                    cruise_height=arch_h,
                    device=ai_device,
                )

                qpos = franka.inverse_kinematics(
                    link=end_effector,
                    pos=target_xyz,
                    quat=down_quat,
                )
                franka.control_dofs_position(qpos[:-2], motors_dof)

                if gripper_mode == "open":
                    franka.control_dofs_position(np.array([0.04, 0.04], dtype=np.float32), fingers_dof)
                else:
                    franka.control_dofs_force(np.array([-1.5, -1.5], dtype=np.float32), fingers_dof)

                # 3 physics steps per rendered frame for crisp tracking and firm grasp contact
                scene.step()
                scene.step()
                scene.step()

                rgb = _to_rgb_numpy(cam.render(rgb=True))

                sim_step_counter += 1
                if sim_step_counter % 140 == 0 or sim_step_counter == 1:
                    print(
                        f"[Genesis Shuffled Sim] Progress: {sim_step_counter}/{total_sim_steps} "
                        f"({sim_step_counter*100//total_sim_steps}%) | {label}: Slot#{task.cube_slot_index}->Bin#{task.bin_slot_index}"
                    )

                annotated = draw_hud_overlay(
                    frame_rgb=rgb,
                    step_idx=sim_step_counter,
                    total_sim_steps=total_sim_steps,
                    phase_title=f"Task {idx+1}/3 ({label}: Slot#{task.cube_slot_index}->Bin#{task.bin_slot_index}) - {sub_name}",
                    status_line=f"Cross-Sorting MLP Active | Gripper: {gripper_mode.upper()}",
                    active_task=task,
                    ee_target=target_xyz,
                    ai_conf=ai_conf,
                    tasks=tasks,
                    completed_count=idx,
                )
                write_rgb_frames(annotated, count=2)

            current_ee_pos = goal_p.copy()

    # 4. Final Summary Card (8.0s = 240 frames)
    summary_card = make_title_card(
        width,
        height,
        "Cross-Order Color Sorting Completed (3/3 Matched)",
        [
            "Shuffled Order Verification (Omnicampus ID: 47412):",
            ">> Task #1 [BLUE ] : Cube Slot #1 (0.46, 0.20) ===> Bin Slot #2 (0.57, -0.26) [SUCCESS]",
            ">> Task #2 [RED  ] : Cube Slot #2 (0.57, 0.24) ===> Bin Slot #3 (0.69, -0.26) [SUCCESS]",
            ">> Task #3 [GREEN] : Cube Slot #3 (0.68, 0.18) ===> Bin Slot #1 (0.45, -0.26) [SUCCESS]",
            "AI Vision accurately matched each cube to its target bin despite shuffled order.",
            "PyTorch MLP Trajectory Policy executed diagonal cross-transport without collision.",
        ],
        "Physical AI Application 1 Final Assignment | Thank you for watching!",
    )
    write_rgb_frames(summary_card, count=240)
    writer.release()

    if os.system(f'ffmpeg -y -loglevel error -i "{temp_mp4}" -vcodec libx264 -pix_fmt yuv420p -movflags +faststart "{output_mp4}"') == 0:
        try:
            os.remove(temp_mp4)
        except OSError:
            pass
    else:
        if os.path.exists(output_mp4):
            os.remove(output_mp4)
        os.rename(temp_mp4, output_mp4)

    print(f"[SUCCESS] Shuffled task video saved to: {os.path.abspath(output_mp4)} (75.0s, 30fps)")
    return os.path.abspath(output_mp4)


if __name__ == "__main__":
    t0 = time.time()
    out_file = "PAI最終課題_47412_shuffled.mp4"
    run_shuffled_simulation_and_record(out_file)
    print(f"Elapsed time: {time.time() - t0:.1f}s")


In [ ]:
# Step 2: 異順列（シャッフル配置）シミュレーションの実行と動画生成
!python run_genesis_sim_shuffled.py

# Step 3: 生成された動画の自動ダウンロード
try:
    from google.colab import files
    files.download('PAI最終課題_47412_shuffled.mp4')
except Exception as e:
    print('自動ダウンロードスキップ:', e)
